# Downstream Training Demo

The official correctness head in this repository is a PyTorch MLP in `code/downstream/train_correctness.py`. This notebook trains a small CPU-only scikit-learn MLP on the same checked-in feature matrices so a beginner can see the training loop without needing a working Torch install.

The idea is the same: train on frozen transition-model features and predict `correctness_score`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent

DATASET = ROOT / "data" / "correctness_dataset"
CANDIDATES = DATASET / "candidates"
FEATURES = DATASET / "features"
HEADS = ROOT / "models" / "correctness_heads"
RESULTS = ROOT / "results" / "analysis"

print(f"Repository root: {ROOT}")

Repository root: /home/bernardod/desktop/research/planfm-validity


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
import warnings

family = "dd_xgb_wl_delta"
seed = 13


def load_split(split):
    data = np.load(FEATURES / family / f"seed_{seed}" / f"{split}.npz", allow_pickle=True)
    X = data["X"].astype(np.float32)
    y = data["correctness_scores"].astype(int)
    return X, y, data

X_train, y_train, train_data = load_split("train")
X_val, y_val, val_data = load_split("validation")
X_test_i, y_test_i, test_i_data = load_split("test-interpolation")
X_test_e, y_test_e, test_e_data = load_split("test-extrapolation")

print(X_train.shape, np.bincount(y_train))

(1156, 55) [925 231]


In [11]:
# Standardize with train-set statistics only.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_i_s = scaler.transform(X_test_i)
X_test_e_s = scaler.transform(X_test_e)

In [12]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    clf = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        batch_size=64,
        learning_rate_init=1e-3,
        max_iter=120,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=13,
    )
    clf.fit(X_train_s, y_train)

print(f"Demo MLP trained for {clf.n_iter_} iterations")

Demo MLP trained for 18 iterations


In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(name, X, y):
    preds = clf.predict(X)
    return {
        "split": name,
        "mae": mean_absolute_error(y, preds),
        "rmse": np.sqrt(mean_squared_error(y, preds)),
        "r2": r2_score(y, preds)
    }

metrics = pd.DataFrame([
    evaluate("validation", X_val_s, y_val),
    evaluate("test-interpolation", X_test_i_s, y_test_i),
    evaluate("test-extrapolation", X_test_e_s, y_test_e),
])
metrics

,split,mae,rmse,r2
0,validation,0.222145,0.315184,0.379119
1,test-interpolation,0.188717,0.278717,0.514479
2,test-extrapolation,0.771991,1.175232,-7.632313


In [19]:
official_command = "python -m code.downstream.train_correctness your_yaml_file.yaml"
print("Official PyTorch training command:")
print(official_command)
print("Check README.md in code.downstream")

Official PyTorch training command:
python -m code.downstream.train_correctness your_yaml_file.yaml
Check README.md in code.downstream
